In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from imblearn.over_sampling import SMOTE

In [3]:
# Load dataset
file_path = 'dataset.csv'
data = pd.read_csv(file_path)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4920 entries, 0 to 4919
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Disease     4920 non-null   object
 1   Symptom_1   4920 non-null   object
 2   Symptom_2   4920 non-null   object
 3   Symptom_3   4920 non-null   object
 4   Symptom_4   4572 non-null   object
 5   Symptom_5   3714 non-null   object
 6   Symptom_6   2934 non-null   object
 7   Symptom_7   2268 non-null   object
 8   Symptom_8   1944 non-null   object
 9   Symptom_9   1692 non-null   object
 10  Symptom_10  1512 non-null   object
 11  Symptom_11  1194 non-null   object
 12  Symptom_12  744 non-null    object
 13  Symptom_13  504 non-null    object
 14  Symptom_14  306 non-null    object
 15  Symptom_15  240 non-null    object
 16  Symptom_16  192 non-null    object
 17  Symptom_17  72 non-null     object
dtypes: object(18)
memory usage: 692.0+ KB


In [5]:
# Step 1: Handle Missing Values
symptom_columns = [f"Symptom_{i}" for i in range(1, 18)]
data[symptom_columns] = data[symptom_columns].fillna("Unknown")

In [6]:
# Encode Diseases
le_disease = LabelEncoder()
data['Disease'] = le_disease.fit_transform(data['Disease'])

In [7]:
# Create binary presence/absence features
binary_features = pd.get_dummies(data[symptom_columns].stack()).groupby(level=0).sum()
y = data['Disease']

In [8]:
# Address Class Imbalance using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(binary_features, y)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [11]:
# Compute sample weights for class imbalance during model training
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Feature Selection
selector = SelectKBest(mutual_info_classif, k=50).fit(X_train, y_train)
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

In [13]:
# Model Initialization
xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)

# Hyperparameter Optimization
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'subsample': [0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3],
    'reg_lambda': [1, 10, 100]
}

random_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_dist, n_iter=50, 
                                   scoring='f1_micro', cv=StratifiedKFold(n_splits=5), n_jobs=-1, random_state=42)
random_search.fit(X_train_selected, y_train, sample_weight=sample_weights)

# Best Model Evaluation
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test_selected)

print("Optimized Classification Report:")
print(classification_report(y_test, y_pred, target_names=le_disease.classes_))

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:24:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Optimized Classification Report:
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        18
                                   AIDS       0.88      0.93      0.90        30
                                   Acne       0.85      0.92      0.88        24
                    Alcoholic hepatitis       1.00      1.00      1.00        25
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        23
                       Bronchial Asthma       1.00      1.00      1.00        33
                   Cervical spondylosis       1.00      1.00      1.00        23
                            Chicken pox       1.00      1.00      1.00        21
                    Chronic cholestasis       1.00      1.00      1.00        15
                            Common Cold       1.00      1.00      1.00     

In [14]:
import joblib

In [15]:
# Save trained model
joblib.dump(best_model, 'xgb_model.pkl')

# Save feature selector
joblib.dump(selector, 'selector.pkl')

# Save label encoder
joblib.dump(le_disease, 'label_encoder.pkl')

# Save binary feature column names
joblib.dump(binary_features.columns.tolist(), 'binary_features_columns.pkl')

print("Model and components saved successfully!")


Model and components saved successfully!
